# Redrob Hackathon - Team Antigravity Sandbox (One-Click Edition)
This notebook executes the exact offline and runtime pipeline on a 100-candidate sample. Just click run on the cell below!

In [ ]:
# 1. Environment Setup
import os
from google.colab import files
import pandas as pd

if os.path.exists("/content/Hack2Skill26"):
    !rm -rf /content/Hack2Skill26

%cd /content
!git clone -b sandbox https://github.com/ADITYASINGH1206/Hack2Skill26.git
%cd /content/Hack2Skill26
!pip install -r requirements.txt -q

# 2. Upload Custom Sample (Optional)
INPUT_FILE = "sample_candidates.jsonl"
print("\n" + "="*60)
print("[Optional] Upload a custom .json or .jsonl sample file (max 100 candidates).")
print("Click 'Cancel Upload' to skip and use the pre-loaded default sample.")
print("="*60)

try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        INPUT_FILE = filename
        print(f"\n[*] Using uploaded file: {INPUT_FILE}")
    else:
        print(f"\n[*] No file uploaded. Defaulting to pre-loaded {INPUT_FILE}")
except Exception:
    print(f"\n[*] Upload skipped or failed. Defaulting to pre-loaded {INPUT_FILE}")

# 3. Phase 1: Data Engineering & Pre-computation
print("\n[*] Starting Phase 1: Wiping old artifacts & generating fresh ones...")
!rm -rf artifacts/*
!python pipeline/ingest_and_filter.py {INPUT_FILE} clean_pool.jsonl
!python pipeline/indexer.py clean_pool.jsonl

# 4. Phase 2: Runtime Inference Engine
print("\n[*] Starting Phase 2: Runtime Inference Engine...")
!python rank.py --out sandbox_output.csv

# 5. Output Validation & Auto-Download
try:
    df = pd.read_csv("sandbox_output.csv")
    print(f"\nTotal Ranked: {len(df)}")
    display(df.head(10))
    print("\n[*] Auto-downloading CSV...")
    files.download("sandbox_output.csv")
except FileNotFoundError:
    print("\n[!] Error: Output file not found. Did the pipeline complete successfully?")